## Question 1:. The Bakersville Bake-off! (3 pts)

The houses in the town of Bakersville are arranged on a grid with R rows (numbered 1 through R) and C columns (numbered 1 through C). The house located on row r and on column c is denoted by (r, c). We say two houses are **neighbors** if they have a common side. 

During the annual Bakersville Bake-off, each household bakes a certain number of cakes to share with their neighbors.
The town comptroller keeps track in an $R \times C$ matrix $A$, where $A_{r, c}$ is the number of cakes which household $(r, c)$ has baked.

The stability of the social order in Bakersville depends on each house baking enough cakes so they can give at least one cake to each neighbor and submit one cake to be judged. Thus, we say a house is **stable** if the number of cakes $A_{r, c}$ is strictly greater than its number of neigbors. We say Bakersville is **stable** if every house is stable. 

The comptroller has asked you to define a function `is_stable` that takes an $R \times C$ numpy array (i.e., an ndarray) as its only argument and returns a Python Boolean that is True if and only if the argument is a stable array and False otherwise. 

In [34]:
import numpy as np

def is_stable(A):
    if A.ndim == 1:
        A = A.reshape(1, -1)

    R, C = A.shape
    for r in range(R):
        for c in range(C):
            neighbors = 0
            if r > 0: neighbors += 1
            if r < R - 1: neighbors += 1
            if c > 0: neighbors += 1
            if c < C - 1: neighbors += 1

            if A[r, c] <= neighbors:
                return False
    return True

In [35]:
array1 = np.array([4, 7, 8,
                   5, 10, 9,
                   3, 5, 7]).reshape(3, 3)
array2 = np.array([3, 2, 4,
                   1, 1, 2,
                   3, 3, 3]).reshape(3, 3)


assert is_stable(array1)
assert not is_stable(array2)
assert not is_stable(np.array([1., 0., 1.]))

## Question 2: Down with `for` loops (12 pts)
Each of the problems below contains a function that uses `for` loops to perform a certain operation on arrays. Your job is to re-write these function using only Numpy array manipulations and library functions (e.g. `np.xxx()`). Do not use any `for` or `while` loops, iterators, generators, or list comprehensions in your solutions.

**2(a)**  Return all of the rows of the 2-D integer matrix `A` where where each entry of the row is distinct (3 pts)

In [36]:
def distinct_rows_py(A):
    'Return all rows of A that have completely distinct entries.'
    return np.array([a for a in A if len(set(a)) == len(a)])
A = np.eye(5)
print(distinct_rows_py(A))
B = np.arange(9).reshape(3, 3)
print(distinct_rows_py(B))

[]
[[0 1 2]
 [3 4 5]
 [6 7 8]]


In [37]:
import numpy as np

def distinct_rows_np(A):
    if A.shape[1] <= 1:
        mask = np.ones(A.shape[0], dtype=bool)
    else:
        sorted_A = np.sort(A, axis=1)
        mask = np.all(np.diff(sorted_A, axis=1) != 0, axis=1)
    return A[mask]

In [38]:
A = np.eye(5)
np.testing.assert_allclose(distinct_rows_np(A), np.empty([0, 5]))

**2(b)** Given a vector $v$ of length $n$, and an integer $0<k<=n$, return a 2D array as: (3 pts)
```
[[v[0], ..., v[k-1]],
 [v[1], ..., v[k]  ],
 [v[2], ..., v[k+1]],
 [ .     .       . ],
 [ .     .       . ],
 [ .     .       . ],
 [v[n-k+1, ..., v[n]]
 ```

In [39]:
def sliding_stack_py(v, k):
    "Stack sliding windows of v of length k."
    rows = []
    for i in range(len(v) - k + 1):
        rows.append(v[i : (i + k)])
    return np.array(rows)
sliding_stack_py(np.array([1, 2, 3, 4, 5]), 3)

array([[1, 2, 3],
       [2, 3, 4],
       [3, 4, 5]])

In [40]:
import numpy as np

def sliding_stack_np(v, k):
    idx = np.arange(len(v) - k + 1)[:, None] + np.arange(k)
    return v[idx]

In [41]:
np.testing.assert_allclose(sliding_stack_np(np.array([1, 2, 3, 4, 5]), 3), np.array([[1, 2, 3],[2, 3, 4],[3, 4, 5]]))
np.testing.assert_allclose(sliding_stack_np(np.array([1, 2, 3, 4, 5]), 5), np.array([[1, 2, 3, 4, 5]]))

**2(c)** Given a vector of non-negative integers `v`, with `max(v) = m`, return a vector `c` of length `m + 1` such that `c[i]` is the number of times that the integer `i` appears in `v`.(3 pts)

In [42]:
def digit_count_py(v):
    m = max(v)
    ret = np.zeros(m + 1, int)
    for vv in v:
        ret[vv] += 1
    return ret
v = np.array([0, 0, 1, 1, 2, 2, 2, 3, 5])
digit_count_py(v)

array([2, 2, 3, 1, 0, 1])

In [43]:
import numpy as np

def digit_count_np(v):
    return np.bincount(v)

In [44]:
v = np.array([0, 0, 1, 1, 2, 2, 2, 3, 4])
np.testing.assert_allclose(digit_count_np(v), np.array([2, 2, 3, 1, 1]))
v = np.array([0, 0, 0, 0, 2, 2, 2, 3, 4])
np.testing.assert_allclose(digit_count_np(v), np.array([4, 0, 3, 1, 1]))

**2(d)** Call a square $n\times n$ matrix $A$ *countersymmetric* if $A_{ij} = A_{n-j,n-i}$ for all $i$ and $j$. An example of such a matrix is:

$$
\begin{pmatrix}
4 & 3 & 2 & 1 & 0\\
8 & 7 & 6 & 5 & 1\\
11 & 10 & 9 & 6 & 2\\
13 & 12 & 10 & 7 & 3\\
14 & 13 & 11 & 8 & 4
\end{pmatrix}
$$

Write a function `is_countersym` that checks this property: (3 pts)

In [45]:
def is_countersym_py(A):
    "Returns True if A is countersymmetric"
    n = A.shape[0]
    for i in range(n):
        for j in range(n):
            if A[i, j] != A[n - j - 1, n - i - 1]:
                return False
    return True
cs_matrix = np.array([[ 4,  3,  2,  1,  0], [ 8,  7,  6,  5,  1], [11, 10,  9,  6,  2], [13, 12, 10,  7,  3], [14, 13, 11,  8,  4]])
is_countersym_py(cs_matrix)

True

In [46]:
import numpy as np

def is_countersym_np(A):
    return bool(np.array_equal(A, np.flip(A).T))

In [47]:
cs_matrix = np.array([[ 4,  3,  2,  1,  0], [ 8,  7,  6,  5,  1], [11, 10,  9,  6,  2], [13, 12, 10,  7,  3], [14, 13, 11,  8,  4]])
assert is_countersym_np(cs_matrix), "The matrix is not counter-symmetric"


## Question 3: $k$-means clustering (14 pts)

$k$-means is a fundamental algorithm for clustering multivariate data. The inputs to the algorithm are:
- An $n\times p$ data matrix $X$ consisting of $n$ observations of a $p$-dimensional feature vector, and
- A $k\times p$ matrix $C$ containing initial guesses for each $k$ cluster centers.

The algorithm proceeds by iteratively a) assigning each point to the nearest cluster center, and b) recomputing the cluster centers as the mean of all of the currently assigned points. Here is a partial implementation:

In [48]:
def kmeans(X, C):
    """
    K-means algorithm.

    Args:
        - X: ndarray, shape (n, p), n observations of p-dimensional feature vector
        - C: ndarray, shape (k, p), k initial cluster centers

    Returns:
        Tuple of length two:
        The first entry is integer ndarray, shape (n), cluster assignments for each data point
        The second entry is ndarray, shape (k, p), centers of each cluster
    """
    assert X.shape[1] == C.shape[1]  # p should match
    old_assignments = None
    while True:
        assignments = nearest_cluster(X, C)
        if np.all(assignments == old_assignments):
            # converged
            return assignments, C
        old_assignments = assignments
        C = compute_centroids(X, assignments)

You will finish implementing this algorithm by completing the missing functions `nearest_cluster()` and `compute_centroids()` below. Note: as in the Question 1, avoid using loops, iterators, or comprehensions. You can find solutions with only broadcasting

**3(a)** Implement the function `nearest_cluster`. It should take two array arguments, the data points `X` and the cluster centers `C`, and return an integer array giving the index in `C` which is nearest to each point in `X`. (3 pts)

In [49]:
def nearest_cluster(X, C):
    """
    For each point in X, find the nearest point in C.

    Args:
        X: ndarray, shape (n, p), n points of dimension p.
        C: ndarray, shape (k, p), k points of dimension p.

    Returns:
        Integer array of length n, [j[1], j[2], ..., j[n]], such that |X[i] - C[j[i]]| <= |X[i] - C[ell]| for 1 <= ell <= k.
    """
    # YOUR CODE HERE
import numpy as np

def nearest_cluster(X, C):

    distances = np.linalg.norm(X[:, None, :] - C, axis=2)

    return np.argmin(distances, axis=1)

In [50]:
ans = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2])
X = np.array([[-1], [0.5], [1], [1.3], [1.6], [2], [2.3], [2.7], [3], [100]])
C = np.array([[1], [2], [3]])
np.testing.assert_allclose(nearest_cluster(X, C), ans)

X = np.array([[-101, 1], [1,1], [1, 1.01], [1.5, 1.6], [2.5, -3], [2.5, 2.7], [3, 33]])
C = np.array([[1, 1], [2,2], [3,3]])
ans = np.array([0, 0, 0, 1, 0, 2, 2])
np.testing.assert_allclose(nearest_cluster(X, C), ans)

**3(b)** Implement the function `compute_centroids`. It should take two array arguments, the data points `X` and the assignment array `a`, and return an $k \times p$ array containing the cluster centroids (averages) for each point assigned to cluster $0, \dots, k-1$. (You may assume that every entry of $a$ is between $0$ and $k-1$, inclusive.) (3 pts) 

In [51]:
import numpy as np

def compute_centroids(X, a):
    k = np.max(a) + 1
    p = X.shape[1]

    sum_X = np.zeros((k, p))
    np.add.at(sum_X, a, X)

    counts = np.bincount(a, minlength=k)

    return sum_X / counts[:, None]

In [52]:
import numpy as np
np.random.seed(1)
X = np.random.rand(30, 2)
a = np.random.randint(0, 5, size=30)
assert compute_centroids(X, a).shape == (5, 2)
assert np.allclose(X[a == 0].mean(axis=0), compute_centroids(X, a)[0])

**3(c)** The performance of the $k$-means algorithm is known to depend heavily on the starting point (the initial clusters `C` passed in as the second argument.) In some cases, using a "good" starting point can dramatically improve the performance of the algorithm.

The $k$-means++ algorithm is designed to find such a good starting point. [According to Wikipedia](https://en.wikipedia.org/wiki/K-means%2B%2B), the steps of $k$-means++ are:

1. Choose one center uniformly at random among the data points.
2. For each data point $x$ not chosen yet, compute $D(x)$, the distance between $x$ and the nearest center that has already been chosen.
3. Choose one new data point at random as a new center, using a weighted probability distribution where a point $x$ is chosen with probability proportional to $D(x)^2$.
4. Repeat Steps 2 and 3 until $k$ centers have been chosen.

Implement this algorithm using the skeleton provided below. As before, your implementation should only use Numpy functions--no additional loops or comprehensions. 

**Note**: To ensure reproducibility, the parts of the algorithm that rely on ranndomness are provided for you. Your job is to fill in the missing lines necessary to complete the algoritm. (5 pts.)

In [53]:
def kmeanspp(X, k, rng):
    n, p = X.shape
    C = np.zeros((k, p))

    j = rng.choice(n)
    C[0] = X[j]
    for i in range(1, k):

        dists_sq = np.min(np.sum((X[:, None, :] - C[:i, :]) ** 2, axis=2), axis=1)
        w = dists_sq / np.sum(dists_sq)

        j = rng.choice(n, p=w)
        C[i] = X[j]
    return C

In [54]:
import numpy as np

np.random.seed(1)
X = np.random.rand(30, 4)
k = 2
rng = np.random.default_rng(12345)
assert kmeanspp(X, k, rng).shape == (2, 4)
ans = np.array(
    [
        [0.87638915, 0.89460666, 0.08504421, 0.03905478],
        [0.6634415, 0.62169572, 0.11474597, 0.94948926],
    ]
)
np.testing.assert_allclose(kmeanspp(X, k, rng), ans)

**3(d)** In order to measure how good a clustering is, we can define the *within-class variance* 

$$ V(\mathbf{X}, \mathbf{a}, \mathbf{C}) = \sum_{i=1}^n \| \mathbf{x}_i - \mathbf{c}_{a_i} \|^2,$$

where the $i$-th element of $\mathbf{a}=\{a_1,\dots,a_n\}$ is the cluster assignment of observation $i$, and $\mathbf{C}=(\mathbf{c}_1,\dots,\mathbf{c}_k)$ are the centers of each cluster. Thus, $V(\mathbf{X}, \mathbf{a}, \mathbf{C})$ is the sum of the squared distance from each data point to the center of its assigned cluster.

Implement this function. (Again, no loops, just use Numpy functions.) (3 pts) 

In [55]:

import numpy as np

def V(X, a, C):
    return np.sum((X - C[a]) ** 2)

In [56]:
import numpy as np

X = np.arange(9).reshape(3, 3)
a = np.arange(3)
C = 2 + X
np.testing.assert_allclose(V(X, a, C), 36)
